In [2]:
import os
import cv2
import gc
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
import mediapipe as mp

# FACE SEGMENTATION ENGINE
def segment_face(image, face_mesh):
    """
    Accepts loaded image, returns cropped face with black background.
    """
    h_img, w_img, _ = image.shape
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_image)

    if not results.multi_face_landmarks:
        return None 

    landmarks = results.multi_face_landmarks[0]
    
    # Get Geometry
    points = np.array([(int(lm.x * w_img), int(lm.y * h_img)) for lm in landmarks.landmark])
    x, y, w_box, h_box = cv2.boundingRect(points)
    
    # Filter: Too Small or Weird Aspect Ratio
    if w_box < 50 or h_box < 50: return None
    aspect = h_box / w_box
    if aspect < 0.6 or aspect > 2.5: return None

    # Masking
    mask = np.zeros((h_img, w_img), dtype=np.uint8)
    hull = cv2.convexHull(points)
    cv2.fillConvexPoly(mask, hull, 255)
    segmented = cv2.bitwise_and(image, image, mask=mask)

    # Cropping with Padding
    pad = 10
    x = max(0, x - pad)
    y = max(0, y - pad)
    w_box = min(w_img, x + w_box + 2*pad) - x
    h_box = min(h_img, y + h_box + 2*pad) - y
    
    return segmented[y:y+h_box, x:x+w_box]

## Delete Files

In [ ]:
import os
import ctypes
import concurrent.futures
from pathlib import Path
from tqdm import tqdm

# Windows DeleteFileW (kernel-level delete)
DeleteFileW = ctypes.windll.kernel32.DeleteFileW
DeleteFileW.argtypes = [ctypes.c_wchar_p]
DeleteFileW.restype = ctypes.c_bool

def fast_delete(path: Path):
    try:
        DeleteFileW(str(path))
    except Exception:
        try:
            os.remove(path)
        except:
            pass

def delete_directory_fast(directory: str, workers: int = 16):
    directory = Path(directory)

    if not directory.exists():
        print(f"Directory does not exist: {directory}")
        return

    print(f"Deleting all files in: {directory}")
    print(f"Using {workers} workers...")

    # Use scandir for extremely fast enumeration
    files = []
    stack = [directory]

    while stack:
        current = stack.pop()
        with os.scandir(current) as it:
            for entry in it:
                if entry.is_file(follow_symlinks=False):
                    files.append(Path(entry.path))
                elif entry.is_dir(follow_symlinks=False):
                    stack.append(Path(entry.path))

    total_files = len(files)
    print(f"Total files detected: {total_files}")

    # Multi-threaded deletion with progress bar
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as exe:
        futures = [exe.submit(fast_delete, f) for f in files]

        with tqdm(total=total_files, desc="Deleting", unit="file") as pbar:
            for _ in concurrent.futures.as_completed(futures):
                pbar.update(1)

    print("File deletion complete.")
    print("Removing empty directories...")

    # Remove empty dirs bottom-up
    for root, dirs, _ in os.walk(directory, topdown=False):
        for d in dirs:
            try:
                os.rmdir(Path(root) / d)
            except:
                pass

    print("Directory cleanup finished.")

# Run
delete_directory_fast(
    directory=r'G:\Thesis\0000_images_remove',
    workers=32
)

Deleting all files in: G:\Thesis\0000_images_remove
Using 32 workers...
Total files detected: 1505207


Deleting:  49%|████▉     | 743322/1505207 [6:47:39<26:47, 473.99file/s]    

## Extract Files

In [ ]:
import zipfile
import os
from pathlib import Path
from tqdm import tqdm

def unzip_fast(zip_path, output_dir):
    zip_path = Path(zip_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {zip_path}")

    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.infolist()

        for member in tqdm(members, desc=f"Extracting {zip_path.name}", unit="item"):

            target_path = output_dir / member.filename

            # Case 1: Directory entry → create the folder
            if member.is_dir():
                target_path.mkdir(parents=True, exist_ok=True)
                continue  # Move to next entry

            # Case 2: File entry → ensure parent exists and extract
            target_path.parent.mkdir(parents=True, exist_ok=True)

            with zf.open(member, 'r') as src, open(target_path, 'wb') as dst:
                # Chunked copy for performance
                while True:
                    chunk = src.read(1024 * 1024)  # 1MB chunks
                    if not chunk:
                        break
                    dst.write(chunk)

    print(f"Completed: {zip_path}")


zip_files = [#r'G:\Thesis\CasualConversationv2_Dataset\Images\CCv2_frames_part_1_0000-1113.zip',
             r'G:\Thesis\CasualConversationv2_Dataset\Images\CCv2_frames_part_2_1114-2227.zip',
             r'G:\Thesis\CasualConversationv2_Dataset\Images\CCv2_frames_part_3_2228-3340.zip',
             #r'G:\Thesis\CasualConversationv2_Dataset\Images\CCv2_frames_part_4_3341-4453.zip',
             #r'G:\Thesis\CasualConversationv2_Dataset\Images\CCv2_frames_part_5_4454-5567.zip'
            ]

for zip_file in zip_files:
    unzip_fast(zip_path=zip_file, output_dir=r"G:\Thesis\CasualConversationv2_Dataset\Images")


Extracting C:\Users\User\Downloads\CCv2_frames_part_2_1114-2227.zip


Extracting CCv2_frames_part_2_1114-2227.zip:   2%|▏         | 1243/54484 [01:15<6:03:16,  2.44item/s]

## Process Monk Skin Tone - Example Dataset

In [ ]:
# CONFIG 
MST_E_ROOT = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\MonkSkinToneDataset\mst-e_data'
MST_E_CSV_RAW = os.path.join(MST_E_ROOT, 'mst-e_image_details.csv')
MST_E_OUTPUT_DIR = r'G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE'

os.makedirs(MST_E_OUTPUT_DIR, exist_ok=True)

# PROCESS MST-E
def process_mste():
    print("\n--- Processing MST-E Dataset ---")
    df = pd.read_csv(MST_E_CSV_RAW)
    df = df.rename(columns={'MST': 'mst_label'})
    
    valid_paths = []
    
    with mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5) as face_mesh:
        
        for i, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
            # Handle Nested Paths: subject_X/image.jpg
            sub_folder = str(row.subject_name)
            fname = str(row.image_ID)
            
            input_path = os.path.join(MST_E_ROOT, sub_folder, fname)
            out_name = f"{sub_folder}_{fname}"
            output_path = os.path.join(MST_E_OUTPUT_DIR, out_name)
            
            if os.path.exists(output_path):
                valid_paths.append(output_path); continue
            
            if not os.path.exists(input_path):
                valid_paths.append(None); continue
                
            try:
                img = cv2.imread(input_path)
                if img is None: valid_paths.append(None); continue
                
                crop = segment_face(img, face_mesh)
                if crop is not None:
                    cv2.imwrite(output_path, crop)
                    valid_paths.append(output_path)
                else:
                    valid_paths.append(None)
            except:
                valid_paths.append(None)
                
            if i % 200 == 0: gc.collect()

    df['segmented_path'] = valid_paths
    df_clean = df.dropna(subset=['segmented_path'])
    out_csv = os.path.join(MST_E_OUTPUT_DIR, 'mst_e_final.csv')
    df_clean.to_csv(out_csv, index=False)
    print(f"MST-E Done. Saved {len(df_clean)} images.")
    return out_csv

# mste_out_csv_path = process_mste()

## Process FACET Dataset

In [3]:
# CONFIG 
FACET_ROOT_IMAGES = r'G:\Thesis\FACET_Dataset\Images' 
FACET_CSV_RAW = r'G:\Thesis\FACET_Dataset\Annotations\annotations\annotations.csv'
consensus_threshold = 0.2
FACET_OUTPUT_DIR = rf'G:\Thesis\FACET_Dataset\Segmented_FACET_{consensus_threshold}'

os.makedirs(FACET_OUTPUT_DIR, exist_ok=True)

# PROCESS FACET
def process_facet(consensus_threshold=0.5, fixed_label=True):
    print("\n--- Processing FACET Dataset ---")
    # Load and filter FACET logic
    df = pd.read_csv(FACET_CSV_RAW)
    skin_cols = [c for c in df.columns if c.startswith("skin_tone_")]
    
    # Filter valid labels
    df = df[df[skin_cols].sum(axis=1) > 0] # Must have votes

    # Calculate Consensus Agreement
    # Find the max votes for the winning class
    max_votes = df[skin_cols].max(axis=1)
    # Find total votes cast
    total_votes = df[skin_cols].sum(axis=1)
    
    # Calculate Agreement Ratio (e.g., 8/10 = 0.8)
    agreement_ratio = max_votes / total_votes
    
    # --- FILTERING STEP ---
    # Only keep rows where agreement >= threshold
    initial_len = len(df)
    df = df[agreement_ratio >= consensus_threshold]
    print(f"Consensus Filter: Dropped {initial_len - len(df)} images with low annotator agreement (<{consensus_threshold*100}%).")
    
    if fixed_label:
        # 4. Assign Final Label
        df["mst_label"] = df[skin_cols].idxmax(axis=1).str.replace("skin_tone_", "")
        
        # 5. Clean up 'na' and convert to int
        df = df[df['mst_label'] != "na"]
        df['mst_label'] = df['mst_label'].astype(int)
    else:
        # Extract numeric class indices from skin_tone_1 ... skin_tone_10
        vote_columns = skin_cols  # already sorted in your CSV

        # Create an array [1, 2, 3, ..., 10] to multiply votes
        class_indices = np.arange(1, len(vote_columns)+1)

        # Example:
        # skin_tone_1  skin_tone_2  ... skin_tone_10
        #       0           2               1
        # continuous = (0*1 + 2*2 + 1*10) / (0+2+1) = 4.67
        df["mst_label"] = df[vote_columns].mul(class_indices).sum(axis=1) / df[vote_columns].sum(axis=1)

        # Drop impossible rows
        df = df.dropna(subset=["mst_label"])

    valid_paths = []
    
    with mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5) as face_mesh:
        
        # Iterating
        for i, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
            fname = row.filename # FACET usually has 'filename' or 'image_id'
            input_path = os.path.join(FACET_ROOT_IMAGES, fname)
            output_path = os.path.join(FACET_OUTPUT_DIR, fname)
            
            if os.path.exists(output_path):
                valid_paths.append(output_path); continue
            
            if not os.path.exists(input_path):
                valid_paths.append(None); continue

            try:
                img = cv2.imread(input_path)
                if img is None: valid_paths.append(None); continue
                
                crop = segment_face(img, face_mesh)
                if crop is not None:
                    cv2.imwrite(output_path, crop)
                    valid_paths.append(output_path)
                else:
                    valid_paths.append(None)
            except:
                valid_paths.append(None)
                
            if i % 500 == 0: gc.collect()

    df['segmented_path'] = valid_paths
    df_clean = df.dropna(subset=['segmented_path'])
    # Add dummy subject name for compatibility
    df_clean['subject_name'] = df_clean['filename'] 
    
    out_csv = os.path.join(FACET_OUTPUT_DIR, 'facet_final.csv')
    df_clean.to_csv(out_csv, index=False)
    print(f"FACET Done. Saved {len(df_clean)} images.")
    return out_csv

# facet_out_csv_path = process_facet(consensus_threshold=consensus_threshold)

FACET_OUTPUT_DIR = rf'G:\Thesis\FACET_Dataset\Segmented_FACET_{consensus_threshold}_continuous'
facet_out_csv_path = process_facet(consensus_threshold=consensus_threshold, fixed_label=False)


--- Processing FACET Dataset ---
Consensus Filter: Dropped 189 images with low annotator agreement (<20.0%).


100%|██████████| 49347/49347 [35:37<00:00, 23.09it/s]  

FACET Done. Saved 3069 images.



C:\Users\User\AppData\Local\Temp\ipykernel_68148\2325165654.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['subject_name'] = df_clean['filename']


## Process Casual Conversation v2 Dataset

In [ ]:
# CONFIG 
CCV2_ROOT = r"G:\Thesis\CasualConversationv2_Dataset\Images"
CCV2_JSON = r"G:\Thesis\CasualConversationv2_Dataset\Annotations\CasualConversationsV2.json"
confidence_filter = None  # Options: None, ["low", "medium", "high"]
CCV2_OUTPUT_DIR = rf"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_{confidence_filter}" if confidence_filter else r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"

os.makedirs(CCV2_OUTPUT_DIR, exist_ok=True)

# PARSE SKIN-TONE LABEL
def parse_mst_label(mst_dict):
    """
    Input: {"scale": "scale 5", "confidence": "medium"}
    Output: mst_label = 5
    """
    scale_str = mst_dict.get("scale", "")
    # Extract last integer
    digits = ''.join([c for c in scale_str if c.isdigit()])
    return int(digits) if digits.isdigit() else None

# PARSE GENDER LABEL
def parse_gender(gender_raw: str):
    """
    CCV2 gender values include:
    - "cis man"
    - "cis woman"
    - "trans man"
    - "trans woman"
    - “nonbinary”, “genderqueer”, etc.

    For your use case:
    Only return "male" or "female".
    Nonbinary / undefined values → return None so they get skipped.
    """
    if not gender_raw:
        return None

    g = gender_raw.strip().lower()

    if "woman" in g or "female" in g:
        return "female"
    elif "man" in g or "male" in g:
        return "male"

    # Nonbinary / other categories → skip by returning None
    return None

def process_casual_conversations_v2(confidence_filter=None):
    print("\n--- Processing Casual Conversations V2 Dataset ---")

    # ---------------------------------------------------------
    # Load JSON and deduplicate by subject_id
    # ---------------------------------------------------------
    with open(CCV2_JSON, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Group by subject_id and take first entry for each subject
    subjects_dict = {}
    
    for entry in data:
        subject_id = entry["subject_id"]
        
        # Skip if we've already seen this subject
        if subject_id in subjects_dict:
            continue
            
        mst = entry.get("monk_skin_tone", {})
        mst_label = parse_mst_label(mst)
        confidence = mst.get("confidence", "").lower()

        if mst_label is None:
            continue

        if confidence_filter is not None:
            if confidence not in confidence_filter:
                continue

        gender_raw = entry.get("gender", "")
        gender_binary = parse_gender(gender_raw)

        # If you want to SKIP nonbinary entries
        # if gender_binary is None:
        #     continue
        
        subjects_dict[subject_id] = {
            "subject_id": subject_id,
            "mst_label": mst_label,
            "confidence": confidence,
            "gender": gender_binary
        }
    
    subject_table = list(subjects_dict.values())
    df_subjects = pd.DataFrame(subject_table)
    
    print(f"Loaded {len(df_subjects)} unique subjects from {len(data)} video entries.")

    # ---------------------------------------------------------
    # Image-level rows (one per image)
    # ---------------------------------------------------------
    image_records = []

    with mp.solutions.face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5
    ) as face_mesh:

        for row in tqdm(df_subjects.itertuples(index=False), total=len(df_subjects)):
            subject_id = row.subject_id
            mst_label = row.mst_label
            confidence = row.confidence
            gender = row.gender

            subj_in_dir = os.path.join(CCV2_ROOT, subject_id)
            if not os.path.isdir(subj_in_dir):
                continue

            subj_out_dir = os.path.join(CCV2_OUTPUT_DIR, subject_id)
            os.makedirs(subj_out_dir, exist_ok=True)

            for fname in os.listdir(subj_in_dir):
                if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue

                # Build full paths
                full_in_path = os.path.join(subj_in_dir, fname)
                cropped_fname = f"{subject_id}_{fname}"
                full_out_path = os.path.join(subj_out_dir, cropped_fname)

                img = cv2.imread(full_in_path)
                if img is None:
                    continue

                crop = segment_face(img, face_mesh)
                if crop is None:
                    continue

                cv2.imwrite(full_out_path, crop)

                image_records.append({
                    "subject_id": subject_id,
                    "mst_label": mst_label,
                    "confidence": confidence,
                    "gender": gender,
                    "original_image": fname,   
                    "cropped_image": cropped_fname
                })

    # ---------------------------------------------------------
    # Save CSV (one row per face image)
    # ---------------------------------------------------------
    df_images = pd.DataFrame(image_records)
    out_csv = os.path.join(CCV2_OUTPUT_DIR, "ccv2_per_image_filenames_only.csv")
    df_images.to_csv(out_csv, index=False)

    print(f"Saved {len(df_images)} rows at: {out_csv}")
    return out_csv

ccv2_out_csv_path = process_casual_conversations_v2(confidence_filter=confidence_filter)


--- Processing Casual Conversations V2 Dataset ---
Loaded 5567 unique subjects from 26467 video entries.


100%|██████████| 5567/5567 [5:14:40<00:00,  3.39s/it]   


Saved 184201 rows at: G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_2\ccv2_per_image_filenames_only.csv
